<a href="https://colab.research.google.com/github/mayankchugh-learning/Advance_RAG/blob/main/HybridSearch_Weaviate_Cohere_ReRanking/HybridSearch_Weaviate_Cohere_15Jun26_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Hybrid Search + Reranking in RAG
### Advanced RAG Series — Episode 02 | **June 2026 — Final Executable**
**by Mayank Chugh | IT AI Enthusiast | @itaienthusiast**

---

## Who is this notebook for?

This notebook is a **hands-on tutorial** for students learning **Retrieval-Augmented Generation (RAG)**. You will build a complete pipeline step by step:

1. **Index** a PDF into **Weaviate Cloud** (vector database)
2. **Retrieve** relevant chunks with **hybrid search** (keyword + semantic)
3. **Rerank** results with **Cohere rerank-v3.5**
4. **Generate** answers with **Qwen3-8B** (quantised LLM)

> **How to use this notebook:** Run cells **top to bottom**. Read each markdown section first — it explains *why* before the code shows *how*. Yellow/warning boxes highlight breaking API changes.

---

## Prerequisites

| Requirement | Details |
|-------------|---------|
| **Runtime** | Google Colab with **T4 GPU** (Runtime → Change runtime type) |
| **Colab Secrets** | `WEAVIATE_URL`, `WEAVIATE_API_KEY`, `HF_TOKEN_READ`, `COHERE_API_KEY` |
| **PDF** | Upload `Retrieval-Augmented-Generation-for-NLP.pdf` to `/content/` |
| **Concepts** | Basic Python, embeddings, and what RAG means at a high level |

---

## What changed in June 2026 (vs older tutorials)

| Component | Model / API | Why it matters |
|-----------|-------------|----------------|
| LLM | `Qwen/Qwen3-8B` | Strong reasoning, Apache 2.0, runs on T4 with 4-bit quantisation |
| Embedding | `BAAI/bge-base-en-v1.5` | Higher quality vectors than `all-MiniLM-L6-v2` |
| Retriever | `WeaviateV4HybridRetriever` (custom) | LangChain's v3 retriever **does not work** with Weaviate v4 |
| Indexing | `collection.batch.dynamic()` | v4 batch API — `retriever.add_documents()` is broken |
| Reranker | `CohereRerank` from `langchain_cohere` | Standalone package; always set `model='rerank-v3.5'` |
| Secrets | `userdata.get()` | Never hard-code API keys in notebook cells |

---

## Live demo results (from saved execution)

Query: *"What is Abstractive Question Answering?"*

**Cohere rerank scores:** Rank 1: **0.6641** · Rank 2: **0.3853** · Rank 3: **0.0860**

> Scores are **not** 0–1 certainties. Lower scores on rank 3 (0.0860) still mean "less relevant" — reranking re-orders chunks so the LLM sees the best context first.

---

## Table of contents

| # | Section | What you learn |
|---|---------|----------------|
| 0 | Pipeline architecture | Big-picture data flow |
| 1 | Install & setup | Required packages |
| 2 | Credentials & Weaviate client | Secure API access, v4 connection |
| 3 | Collection & custom retriever | Schema, hybrid search, `alpha` |
| 4 | Qwen3-8B LLM | 4-bit loading, generation settings |
| 5 | Load PDF & index | Chunking, batch upload |
| 6 | Baseline RAG chain | Retrieve → generate (no rerank) |
| 7 | Cohere reranking | Cross-encoder rescoring |
| 8 | Reranked RAG chain | Full improved pipeline |
| 9 | Results & takeaways | Compare baseline vs reranked |


---
# Section 0 — Pipeline Architecture

Read this diagram **before** running code. Every later section maps to one of these three phases.

```
┌─────────────────────────────────────────────────────────────────────────┐
│  PHASE 1 — Setup & Index                                                │
│  • Connect to Weaviate Cloud (v4 API) using Colab Secrets               │
│  • Create collection "RAG" with text2vec-huggingface embedder           │
│  • Embedding model: BAAI/bge-base-en-v1.5                               │
│  • Custom WeaviateV4HybridRetriever (alpha=0.5)                         │
│  • Index PDF pages via collection.batch.dynamic()                       │
├─────────────────────────────────────────────────────────────────────────┤
│  PHASE 2 — Hybrid Retrieval                                             │
│  • User query → BM25 (keyword) + dense vector (semantic) search         │
│  • Weaviate fuses both with RRF → returns top-K=4 document chunks       │
│  • alpha=0.5 → equal weight to keyword vs semantic (tune 0.0–1.0)       │
├─────────────────────────────────────────────────────────────────────────┤
│  PHASE 3 — Rerank & Generate                                            │
│  • Cohere rerank-v3.5 rescores top-K → keeps top_n=3 best chunks        │
│  • Live demo scores: 0.6641 / 0.3853 / 0.0860                           │
│  • Qwen3-8B (NF4 4-bit) generates answer from reranked context          │
└─────────────────────────────────────────────────────────────────────────┘
```

### Key vocabulary

| Term | Meaning |
|------|---------|
| **Hybrid search** | Combines BM25 (exact word match) with vector search (meaning match) |
| **alpha** | Weaviate knob: 0 = pure BM25, 1 = pure vector, 0.5 = balanced |
| **Reranking** | Second-stage model scores query–document pairs; fixes wrong order from retrieval |
| **Stuff chain** | LangChain puts all retrieved chunks into one prompt (`chain_type='stuff'`) |

---


---
# Section 1 — Install Dependencies

Run the next two cells to install packages. Each line in the `pip install` command serves a purpose:

| Package | Purpose in this notebook |
|---------|--------------------------|
| `weaviate-client` | Weaviate v4 Python client (v3 API removed Dec 2024) |
| `langchain`, `langchain_community`, `langchain_classic` | RAG chains, PDF loader, HF pipeline wrapper |
| `langchain-cohere` | `CohereRerank` compressor |
| `pypdf` | Load the RAG paper PDF |
| `bitsandbytes`, `accelerate` | 4-bit quantisation for Qwen3 on T4 GPU |
| `transformers>=4.51.0` | Qwen3 model support |
| `cohere` | Underlying Cohere SDK |

---


In [3]:
# Install all dependencies for the hybrid RAG + reranking pipeline
# weaviate-client  → v4 API only (v3 removed Dec 2024)
# transformers>=4.51.0 → required for Qwen3 architecture support
# langchain-cohere → CohereRerank lives in this standalone package (not core langchain)
!pip install -q weaviate-client langchain langchain_community langchain_classic \
             pypdf bitsandbytes accelerate cohere langchain-cohere \
             "transformers>=4.51.0"


In [4]:
# Pin LangChain packages to compatible versions (resolves occasional Colab dependency drift)
!pip install -q --upgrade langchain langchain_community


---
# Section 1 — Credentials & Weaviate Client (v4 API)

### What is hybrid search?

When a user asks *"What is abstractive QA?"*, a **vector-only** retriever might miss chunks that use the exact phrase *"MSMARCO abstractive QA"*. A **keyword-only** (BM25) retriever might miss paraphrases. **Hybrid search runs both** and merges rankings — better recall for RAG.

### Set up Colab Secrets (do this once)

1. Click the **key icon** in Colab's left sidebar → **Secrets**
2. Add each secret (names must match exactly):

| Secret name | Used for |
|-------------|----------|
| `WEAVIATE_URL` | Weaviate Cloud cluster URL |
| `WEAVIATE_API_KEY` | Weaviate authentication |
| `HF_TOKEN_READ` | HuggingFace inference for embeddings at query time |
| `COHERE_API_KEY` | Cohere rerank API |

3. Toggle **Notebook access** ON for each secret

> **Security:** Never paste API keys directly in code cells. `userdata.get('SECRET_NAME')` reads from Colab Secrets at runtime.

> **API change:** `weaviate.Client()` was removed in Dec 2024. This notebook uses `weaviate.connect_to_weaviate_cloud()`.

---


In [5]:
# Core Weaviate v4 imports — Auth wraps your API key for cloud connection
import weaviate
from weaviate.classes.init import Auth
import os


In [6]:
# Colab-only: reads secrets from the sidebar key icon (not available outside Colab)
from google.colab import userdata


### Step A — Load credentials (explore mode)

The next few cells load secrets **one at a time** so you can see each variable. In practice, use the **consolidated cell** further below that loads all keys together.

---


In [7]:
# Step 1 of 2: Weaviate cluster credentials (see consolidated cell below for all keys)
WEAVIATE_URL = userdata.get('WEAVIATE_URL')
WEAVIATE_API_KEY = userdata.get('WEAVIATE_API_KEY')


In [8]:
# Step 2 of 2: HuggingFace token for embedding API calls at query/index time
HF_TOKEN = userdata.get('HF_TOKEN_READ')


In [9]:
# Consolidated credentials — use this cell when re-running the notebook
WEAVIATE_URL = userdata.get('WEAVIATE_URL')
WEAVIATE_API_KEY = userdata.get('WEAVIATE_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN_READ')
COHERE_API_KEY = userdata.get('COHERE_API_KEY')


In [10]:
# Connect to Weaviate Cloud (v4) WITH HuggingFace header
# The HF header is required whenever text2vec-huggingface must embed text (queries + indexing)
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
    headers={'X-HuggingFace-Api-Key': HF_TOKEN},
)
client.is_ready()  # Expected output: True


True

---
# Section 2 — Collection & WeaviateV4HybridRetriever

### Create the vector store

We define a Weaviate **collection** (like a table) with one text field: `content`. Weaviate's `text2vec-huggingface` module embeds each chunk automatically when we batch-upload documents.

### Why a custom retriever class?

`WeaviateHybridSearchRetriever` from LangChain Community targets the **old v3 client**. Our Weaviate cluster uses **v4**, so we implement a thin wrapper around `collection.query.hybrid()`.

### Embedding upgrade

| Old | New | Benefit |
|-----|-----|---------|
| `all-MiniLM-L6-v2` | `BAAI/bge-base-en-v1.5` | Higher MTEB benchmark scores → better semantic retrieval |

### The `alpha` parameter

- `alpha=0.0` → keyword (BM25) only  
- `alpha=1.0` → vector only  
- `alpha=0.5` → **balanced** (used in this demo)

---


In [11]:
# Clean slate: delete old collection so schema/embedding model changes take effect
if client.collections.exists('RAG'):
    client.collections.delete('RAG')
    print('Deleted existing RAG collection')
else:
    print('No existing RAG collection — starting fresh')


Deleted existing RAG collection


In [12]:
# Alternate client without HF header — sufficient for schema-only operations (create/delete collection)
import os
weaviate_url = WEAVIATE_URL
weaviate_api_key = WEAVIATE_API_KEY
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=Auth.api_key(weaviate_api_key),
)
print(client.is_ready())


True


In [13]:
from weaviate.classes.config import Configure, Property, DataType

client.collections.delete('RAG')  # Ensure clean slate

# Schema: single TEXT property 'content' — Weaviate embeds this field automatically
client.collections.create(
    name='RAG',
    description='Documents for RAG',
    vectorizer_config=Configure.Vectorizer.text2vec_huggingface(
        model='BAAI/bge-base-en-v1.5',   # Upgraded from all-MiniLM-L6-v2
        vectorize_collection_name=False,
    ),
    properties=[
        Property(
            name='content',
            data_type=DataType.TEXT,
            description='The content of the document chunk',
            skip_vectorization=False,
            vectorize_property_name=False,
        )
    ],
)
print('RAG collection created with BAAI/bge-base-en-v1.5 embedding model')


RAG collection created with BAAI/bge-base-en-v1.5 embedding model


In [14]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from typing import List, Any
import weaviate.classes.query

# Custom retriever: LangChain's WeaviateHybridSearchRetriever does NOT support v4 client
# This class calls collection.query.hybrid() and returns LangChain Document objects
class WeaviateV4HybridRetriever(BaseRetriever):
    client: Any
    index_name: str
    text_key: str = 'content'
    alpha: float = 0.5      # 0=BM25 only, 1=vector only, 0.5=balanced hybrid
    attributes: List[str] = []
    k: int = 4              # number of chunks to retrieve per query

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(self, query: str, *, run_manager: Any = None) -> List[Document]:
        collection = self.client.collections.get(self.index_name)
        return_props = list(set(self.attributes + [self.text_key]))
        response = collection.query.hybrid(
            query=query,
            alpha=self.alpha,
            return_properties=return_props,
            return_metadata=weaviate.classes.query.MetadataQuery(score=True),
            limit=self.k,
        )
        docs = []
        for obj in response.objects:
            properties = obj.properties or {}
            page_content = properties.pop(self.text_key, '')
            metadata = {}
            if obj.metadata and obj.metadata.score is not None:
                metadata['score'] = obj.metadata.score
            metadata.update(properties)
            docs.append(Document(page_content=page_content, metadata=metadata))
        return docs

# Instantiate with alpha=0.5 (equal BM25 + semantic weighting)
retriever = WeaviateV4HybridRetriever(
    client=client,
    index_name='RAG',
    text_key='content',
    alpha=0.5,
    attributes=[],
    k=4
)


/tmp/ipykernel_2430/1427478755.py:8: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class WeaviateV4HybridRetriever(BaseRetriever):


---
# Section 3 — Qwen3-8B (NF4 4-bit Quantised)

The **generator** in our RAG pipeline. It reads retrieved (and later reranked) context and writes the final answer.

| Setting | Value | Why |
|---------|-------|-----|
| Model | `Qwen/Qwen3-8B` | Strong open-weight model, Apache 2.0 |
| Quantisation | NF4 4-bit via `bitsandbytes` | Fits on Colab T4 (~5 GB VRAM) |
| `do_sample=False` | Greedy decoding | Deterministic RAG answers; **required for Qwen3 non-thinking mode** |

> **Warning:** Do **not** pass `temperature`, `top_k`, or `top_p` — Qwen3 ignores them and prints warnings.

> **Deprecation note:** `HuggingFacePipeline` from `langchain_community` still works but LangChain recommends migrating to `langchain_huggingface` long term.

---


In [15]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_community.llms import HuggingFacePipeline
# langchain-community sunset warning is expected — safe to proceed for this tutorial


/tmp/ipykernel_2430/2835339988.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


In [16]:
# Generator model: Qwen3-8B replaces older zephyr-7b-beta from prior notebook versions
model_name = 'Qwen/Qwen3-8B'


In [17]:
def load_quantized_model(model_name: str):
    # NF4 4-bit quantisation — fits Qwen3-8B on Colab T4 GPU
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        quantization_config=bnb_config,
        device_map='auto',
    )
    return model


In [18]:
def initialize_tokenizer(model_name: str):
    # Qwen3 uses its own tokenizer defaults — no bos_token override needed (unlike Zephyr)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return tokenizer


In [19]:
tokenizer = initialize_tokenizer(model_name)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [20]:
model = load_quantized_model(model_name)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [21]:
# do_sample=False → greedy / non-thinking mode (best for factual RAG answers)
# Do NOT pass temperature, top_k, or top_p — Qwen3 ignores them
text_pipeline = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device_map='auto',
    do_sample=False,
    max_new_tokens=512,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'num_return_sequences', 'max_new_tokens', 'pad_token_id', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [22]:
# Wrap HuggingFace pipeline as a LangChain LLM for RetrievalQA
llm = HuggingFacePipeline(pipeline=text_pipeline)


/tmp/ipykernel_2430/326805344.py:1: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_pipeline)


---
# Section 4 — Load PDF & Index Documents

### Document source

We use the original **RAG paper** PDF (`Retrieval-Augmented-Generation-for-NLP.pdf`). Each page becomes one LangChain `Document`; Weaviate stores the page text as a searchable chunk.

### Critical v4 indexing pattern

1. **Reconnect** the Weaviate client **with** the `X-HuggingFace-Api-Key` header (needed for embedding at index time)
2. Use `collection.batch.dynamic()` — **not** `retriever.add_documents()` (broken on v4)
3. Only pass properties defined in the schema (`content` only)

### Sanity check

After indexing, we run a test query (`"what is RAG token?"`) and print the first retrieved chunk to confirm hybrid search works.

---


In [23]:
from langchain_community.document_loaders import PyPDFLoader

# Upload Retrieval-Augmented-Generation-for-NLP.pdf to /content/ before running
doc_path = '/content/Retrieval-Augmented-Generation-for-NLP.pdf'
loader = PyPDFLoader(doc_path)
docs = loader.load()
print(f'Loaded {len(docs)} pages')


Loaded 19 pages


### Inspect a sample page (optional)

The cell below prints **page 7** (index 6) of the PDF. This page contains **Figure 2** from the RAG paper — the famous *RAG-Token* example showing how different retrieved documents activate for different generated tokens (Hemingway → *A Farewell to Arms* vs *The Sun Also Rises*).

---


In [24]:
# Preview one page — see markdown above for why page 7 (RAG-Token figure) matters
docs[6]


Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/Retrieval-Augmented-Generation-for-NLP.pdf', 'total_pages': 19, 'page': 6, 'page_label': '7'}, page_content='Document 1: his works are considered classics of American\nliterature ... His wartime experiences formed the basis for his novel\n”A Farewell to Arms” (1929) ...\nDocument 2: ... artists of the 1920s ”Lost Generation” expatriate\ncommunity . His debut novel, ”The Sun Also Rises”, was published\nin 1926.\nBOS\n”\nTheSunAlso\nR ises\n” is a\nnovel\nby this\nauthor\nof ” A\nFarewellto\nArms\n”\nDoc 1\nDoc 2\nDoc 3\nDoc 4\nDoc 5\nFigure 2: RAG-Token document posteriorp(zi|x,yi,y−i) for each generated token for inpu

In [25]:
# v4 batch indexing — client MUST include X-HuggingFace-Api-Key header for embedding
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
    headers={'X-HuggingFace-Api-Key': HF_TOKEN},  # Required for text2vec-huggingface
)

collection = client.collections.get(retriever.index_name)

# batch.dynamic() replaces broken retriever.add_documents() on Weaviate v4
with collection.batch.dynamic() as batch_writer:
    for doc in docs:
        batch_writer.add_object(properties={retriever.text_key: doc.page_content})

print('Documents indexed in Weaviate with BAAI/bge-base-en-v1.5 embeddings')


Documents indexed in Weaviate with BAAI/bge-base-en-v1.5 embeddings


In [26]:
# Reload credentials; HF_TOKEN_WRITE used if your HF account needs write scope for some ops
WEAVIATE_URL = userdata.get('WEAVIATE_URL')
WEAVIATE_API_KEY = userdata.get('WEAVIATE_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN_READ')
COHERE_API_KEY = userdata.get('COHERE_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN_WRITE')


In [27]:
# Point retriever at the HF-header client so query-time embedding works
retriever = WeaviateV4HybridRetriever(
    client=client,
    index_name='RAG',
    text_key='content',
    alpha=0.5,
    attributes=[],
    k=4
)

test_result = retriever.invoke('what is RAG token?')
print(f'Retrieved {len(test_result)} documents')  # Expected: 4
if test_result:
    print(test_result[0].page_content[:300])


Retrieved 4 documents
byθ that generates a current token based on a context of the previousi− 1 tokensy1:i−1, the original
inputx and a retrieved passagez.
To train the retriever and generator end-to-end, we treat the retrieved document as a latent variable.
We propose two models that marginalize over the latent document


---
# Section 5 — Baseline Hybrid RAG Chain (No Reranking)

This is our **before** picture: hybrid retrieval → LLM, with **no reranking**.

```
User question → WeaviateV4HybridRetriever (top 4) → Qwen3-8B → Answer
```

### What to expect

- Qwen3 may echo long context before answering — this is normal for the baseline chain
- Retrieved chunks are ordered by hybrid score, not by deep query–passage relevance
- Compare this output with Section 8 (reranked chain) to see reranking's effect

---


In [28]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate


In [29]:
# Prompt template: instructs LLM to answer only from provided context
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, say you don't know.
Keep the answer concise and factual.

{context}
Question: {question}
Helpful Answer:"""

prompt = PromptTemplate.from_template(template)


In [30]:
# Baseline chain: hybrid retriever → stuff all chunks into prompt → Qwen3 generates
hybrid_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
)


In [31]:
# Baseline: no reranking — compare this verbose output with Section 7 reranked chain
result = hybrid_chain.invoke({'query': 'What is Abstractive Question Answering?'})
print('=== BASELINE (no reranking) ===')
print(result['result'])


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set cle

=== BASELINE (no reranking) ===
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

MSMARCO as an open-domain abstractive QA task. MSMARCO has some questions that cannot be
answered in a way that matches the reference answer without access to the gold passages, such as
“What is the weather in V olcano, CA?” so performance will be lower without using gold passages.
We also note that some MSMARCO questions cannot be answered using Wikipedia alone. Here,
RAG can rely on parametric knowledge to generate reasonable responses.
3.3 Jeopardy Question Generation
To evaluate RAG’s generation abilities in a non-QA setting, we study open-domain question gen-
eration. Rather than use questions from standard open-domain QA tasks, which typically consist
of short, simple questions, we propose the more demanding task of generating Jeopardy questions.
Jeopardy is an unusual format that consis

---
# Section 6 — Cohere Reranking (rerank-v3.5)

### Why rerank after hybrid retrieval?

Hybrid search is **fast** but uses **bi-encoder** scores (query and document encoded separately). **Rerankers** are **cross-encoders** — they read query + document together and produce a sharper relevance score. Typical pattern:

```
Retrieve many (fast, cheap) → Rerank few (slow, accurate) → Generate
```

### Implementation

| Piece | Class | Role |
|-------|-------|------|
| Base retriever | `WeaviateV4HybridRetriever` | Gets top-K=4 candidates |
| Compressor | `CohereRerank` | Scores and keeps top_n=3 |
| Wrapper | `ContextualCompressionRetriever` | Chains retriever + reranker |

> Import from **`langchain_cohere`** — not the old `langchain.retrievers.document_compressors` path.

> Always set `model='rerank-v3.5'` explicitly.

### Live demo scores (query: Abstractive QA)

Rank 1: **0.6641** · Rank 2: **0.3853** · Rank 3: **0.0860**

Rank 1 (0.6641) is the MSMARCO abstractive QA section — exactly what we want. Rank 3 (0.0860) is a figure/diagram chunk — correctly demoted.

---


In [33]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank  # Standalone package — NOT langchain.retrievers.document_compressors


In [34]:
# Cohere cross-encoder reranker — always specify model explicitly
compressor = CohereRerank(
    cohere_api_key=COHERE_API_KEY,
    model='rerank-v3.5',
    top_n=3,
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)


In [35]:
user_query = 'What is Abstractive Question Answering?'
compressed_docs = compression_retriever.invoke(user_query)
print(f'Reranked documents returned: {len(compressed_docs)}\n')
for i, doc in enumerate(compressed_docs):
    score = doc.metadata.get('relevance_score', 'N/A')
    print(f'Rank {i+1} | Relevance score: {score:.4f}')
    print(doc.page_content[:200])
    print('---')
# Saved execution scores: Rank 1=0.6641, Rank 2=0.3853, Rank 3=0.0860


Reranked documents returned: 3

Rank 1 | Relevance score: 0.6641
minimize the negative marginal log-likelihood of each target,∑
j− logp(yj|xj) using stochastic
gradient descent with Adam [28]. Updating the document encoder BERTd during training is costly as
it requ
---
Rank 2 | Relevance score: 0.3853
MSMARCO as an open-domain abstractive QA task. MSMARCO has some questions that cannot be
answered in a way that matches the reference answer without access to the gold passages, such as
“What is the w
---
Rank 3 | Relevance score: 0.0860
The	Divine
Comedy	(x) q
Query
Encoder
q(x)
MIPS pθ
Generator pθ
(Parametric)
Margin-
alize
This	14th	century	work
is	divided	into	3
sections:	"Inferno",
"Purgatorio"	&
"Paradiso"									(y)
End-to-En
---


---
# Section 7 — Reranked RAG Chain

Same `RetrievalQA` chain as the baseline, but the retriever is now `compression_retriever` (hybrid + Cohere rerank).

```
User question → Hybrid retrieve (4) → Cohere rerank (3) → Qwen3-8B → Answer
```

Run both demo queries and compare with Section 5:

| Query | What to observe |
|-------|-----------------|
| *What is Abstractive Question Answering?* | Reranked context should prioritise Section 3.2 / MSMARCO |
| *what is RAG token?* | Should surface RAG-Token model description (Figure 2 page) |

---


In [36]:
# Same RetrievalQA chain, but retriever now includes Cohere reranking step
hybrid_chain_reranked = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=compression_retriever,
)


In [37]:
response = hybrid_chain_reranked.invoke({'query': 'What is Abstractive Question Answering?'})
print('=== RERANKED (Cohere rerank-v3.5 + Qwen3-8B) ===')
print(response.get('result'))


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


=== RERANKED (Cohere rerank-v3.5 + Qwen3-8B) ===
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

minimize the negative marginal log-likelihood of each target,∑
j− logp(yj|xj) using stochastic
gradient descent with Adam [28]. Updating the document encoder BERTd during training is costly as
it requires the document index to be periodically updated as REALM does during pre-training [20].
We do not ﬁnd this step necessary for strong performance, and keep the document encoder (and
index) ﬁxed, only ﬁne-tuning the query encoder BERTq and the BART generator.
2.5 Decoding
At test time, RAG-Sequence and RAG-Token require different ways to approximatearg maxyp(y|x).
RAG-Token The RAG-Token model can be seen as a standard, autoregressive seq2seq genera-
tor with transition probability: p′
θ(yi|x,y 1:i−1) = ∑
z∈top-k(p(·|x))pη(zi|x)pθ(yi|x,zi,y 1:i−1) To
decode, we can plugp′
θ(yi|x,

In [38]:
response2 = hybrid_chain_reranked.invoke({'query': 'what is RAG token?'})
print('=== RERANKED (Cohere rerank-v3.5 + Qwen3-8B) ===')
print(response2.get('result'))


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


=== RERANKED (Cohere rerank-v3.5 + Qwen3-8B) ===
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

byθ that generates a current token based on a context of the previousi− 1 tokensy1:i−1, the original
inputx and a retrieved passagez.
To train the retriever and generator end-to-end, we treat the retrieved document as a latent variable.
We propose two models that marginalize over the latent documents in different ways to produce a
distribution over generated text. In one approach, RAG-Sequence, the model uses the same document
to predict each target token. The second approach, RAG-Token, can predict each target token based
on a different document. In the following, we formally introduce both models and then describe the
pη andpθ components, as well as the training and decoding procedure.
2.1 Models
RAG-Sequence Model The RAG-Sequence model uses the same retrieved document to g

---
# Section 8 — Results & Takeaways

Use this section as a **self-check** after running the notebook.

---


### Key lessons (verified against live execution)

1. **Weaviate v4 needs a custom retriever** — `WeaviateHybridSearchRetriever` fails with the v4 client; use `WeaviateV4HybridRetriever`.
2. **Index with `collection.batch.dynamic()`** — reconnect client with HF header first.
3. **CohereRerank lives in `langchain_cohere`** — always pass `model='rerank-v3.5'`.
4. **Rerank scores are relative, not absolute** — live demo: 0.6641 / 0.3853 / 0.0860. A low rank-3 score still means "discard this chunk."
5. **Qwen3-8B: use `do_sample=False` only** — other sampling params are ignored.
6. **Baseline vs reranked** — baseline may dump irrelevant pages; reranking lifts the best passages to the top of the prompt.

> **Next episode:** custom reranking from scratch + FlashRank as a free local alternative.

---


## Pipeline summary — final execution (June 2026)

| Component | Class / API | Model / value | Role in pipeline |
|-----------|-------------|---------------|-------------------|
| Vector DB | Weaviate Cloud v4 | — | Stores and searches document chunks |
| Embedding | `text2vec-huggingface` | `BAAI/bge-base-en-v1.5` | Dense vectors for semantic leg of hybrid search |
| Retriever | `WeaviateV4HybridRetriever` | `alpha=0.5`, `k=4` | BM25 + vector fusion |
| Indexing | `collection.batch.dynamic()` | — | v4-compatible bulk upload |
| Reranker | `CohereRerank` | `rerank-v3.5`, `top_n=3` | Cross-encoder rescoring |
| Live rerank scores | — | **0.6641 / 0.3853 / 0.0860** | From saved notebook execution |
| LLM | `HuggingFacePipeline` | `Qwen/Qwen3-8B` NF4 | Answer generation |
| Chain | `RetrievalQA` | `chain_type='stuff'` | Wires retriever + LLM |

---
*Made with care by Mayank Chugh | GitHub: [mayankchugh-learning](https://github.com/mayankchugh-learning) | @itaienthusiast*
